In [0]:
import requests
import json
import time
from datetime import datetime, timedelta, timezone

In [0]:
# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("schema", "valeriimatviiv_bronze", "2. Target Schema")
dbutils.widgets.text("volume", "market_radar_landing", "3. Landing Volume")
dbutils.widgets.text("tickers", "AAPL,NVDA,MSFT,AMZN,TSLA", "4. Tickers")
dbutils.widgets.text("secret_scope", "valerii-matviiv-scope", "5. Secret Scope")
dbutils.widgets.text("secret_key", "finnhub-api-key", "6. Secret Key")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
tickers = [t.strip() for t in dbutils.widgets.get("tickers").split(",")]
secret_scope = dbutils.widgets.get("secret_scope")
secret_key = dbutils.widgets.get("secret_key")

api_key = dbutils.secrets.get(scope=secret_scope, key=secret_key)

landing_path = f"/Volumes/{catalog}/{schema}/{volume}/landing/finnhub_news"
dbutils.fs.mkdirs(landing_path)


In [0]:
# Define 7-day rolling window
now = datetime.now(timezone.utc)
from_date = (now - timedelta(days=7)).strftime("%Y-%m-%d")
to_date = now.strftime("%Y-%m-%d")

all_news = []

# Fetch company-specific news
for symbol in tickers:
    url = f"https://finnhub.io/api/v1/company-news?symbol={symbol}&from={from_date}&to={to_date}&token={api_key}"
    response = requests.get(url)
    if response.status_code == 200:
        articles = response.json()
        for article in articles:
            article["related"] = symbol
            article["_schema_phase"] = "nasdaq100_company"
            all_news.append(article)
    time.sleep(0.2)

# Fetch general market news
gen_url = f"https://finnhub.io/api/v1/news?category=general&token={api_key}"
gen_resp = requests.get(gen_url)
if gen_resp.status_code == 200:
    gen_articles = gen_resp.json()
    for article in gen_articles:
        article["_schema_phase"] = "general_market"
        all_news.append(article)

output_file = f"{landing_path}/finnhub_7d_news.json"
with open(output_file, "w") as f:
    json.dump(all_news, f)

# print(f"Successfully landed {len(all_news)} 7-day news articles to: {output_file}")

In [0]:
# import json

# output_file = f"/Volumes/{catalog}/{schema}/{volume}/landing/finnhub_news/finnhub_7d_news.json"
# with open(output_file, "r") as f:
#     data = json.load(f)

# print(f"Total News Articles Retained: {len(data)}")
# if len(data) > 0:
#     print("Sample Article Structure:", json.dumps(data[0], indent=2))